In [1]:
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

# Check 1: what tickers exist
cur.execute("SELECT DISTINCT ticker FROM filing_chunks ORDER BY ticker;")
print("Tickers in DB:", cur.fetchall())

# Check 2: raw FTS with ticker filter
cur.execute("""
    SELECT COUNT(*) FROM filing_chunks
    WHERE ticker = 'AAPL'
    AND fts @@ plainto_tsquery('english', 'cybersecurity')
""")
print("AAPL cybersecurity matches:", cur.fetchone()[0])

# Check 3: what query string is actually hitting FTS
query = "What are Apple's cybersecurity risks?"
cur.execute("""
    SELECT COUNT(*) FROM filing_chunks
    WHERE ticker = 'AAPL'
    AND fts @@ plainto_tsquery('english', %s)
""", (query,))
print("With full question string:", cur.fetchone()[0])

cur.close()
conn.close()

Tickers in DB: [('AAPL',), ('GOOGL',), ('META',), ('MSFT',), ('NVDA',)]
AAPL cybersecurity matches: 3
With full question string: 1
